# 18 — สอนวิเคราะห์ Rate Ratio (อัตราส่วนอัตราอุบัติการณ์) ด้วยข้อมูลจริงระดับตำบล จังหวัดสระบุรี

**คำถาม:** ประชากรในตำบลที่มี **แหล่งกำเนิด PM2.5 สูง** (อุตสาหกรรม/เหมือง, ถนนสายหลัก, การเผา) มี **อัตราป่วยด้วยโรคที่เกี่ยวกับ PM2.5 สูงกว่า** ตำบลที่มีแหล่งกำเนิดต่ำกี่เท่า

| ส่วน | เนื้อหา |
|---|---|
| 2 | อัตรา (rate), person-time, **Rate Ratio (RR)**, Rate Difference, CI แบบ Wald และแบบ exact |
| 3 | RR แบบหยาบ (crude) และ **dose–response** (แบ่ง 3 ระดับ + trend test) |
| 4 | ควบคุมตัวกวน: **Mantel–Haenszel RR** + การทดสอบความเป็นเอกพันธ์ (homogeneity) |
| 5 | **Poisson regression มี offset** → adjusted IRR, ตรวจ overdispersion → Negative binomial / cluster-robust SE |
| 6 | Forest plot: adjusted IRR รายโรค |
| 7 | ข้อจำกัด (ecological fallacy ฯลฯ) และแบบฝึกหัด |

**ข้อมูลจริง (ดึงอัตโนมัติ ไม่ต้องอัปโหลดไฟล์)**

| ข้อมูล | แหล่ง |
|---|---|
| จำนวนผู้ป่วย 11 โรค รายตำบล × ปีงบประมาณ | **MOPH Open Data API** ตาราง `s_pm25_disease` (ข้อมูล HDC) |
| ประชากร (ตัวหาร → person-years) | WorldPop 2020 |
| ขอบเขตตำบล + รหัสตำบล | OCHA COD-AB Thailand (สำรอง GADM/OSM) |
| การสัมผัส (exposure) | OSM อุตสาหกรรม/เหมือง, OSM ถนนสายหลัก, NASA FIRMS จุดความร้อน |

> ⚠ เว็บกระทรวงสาธารณสุขอาจไม่ตอบสนองต่อ IP ของ Colab — ถ้าเกิดขึ้น โค้ดจะแจ้งและให้เลือกอ่านไฟล์ที่ได้จากสคริปต์ `สคริปต์PYดึงข้อมูลผู้ป่วยรายตำบลจากHDC.py (Drive: 3-ColabProjects/Training)` ใน Drive แทน (ตั้ง `HDC_FALLBACK_CSV`)

## 0. CONFIG

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
PROVINCE_EN = "Saraburi"
HDC_YEARS = list(range(2559, 2570))              # ปีงบ พ.ศ. 2559–2569 (= ค.ศ. 2016–2026) — ปีที่ไม่มี/ไม่ครบจะถูกตัดออกพร้อมแจ้ง
YEARS = list(range(2016, 2027))                  # ปี ค.ศ. ของจุดความร้อน FIRMS 2016–2026 (ใช้สร้างตัวแปร exposure)
BOUNDARY_SOURCE = "drive"       # "drive" (shapefile ตำบลใน Google Drive) → สำรอง "cod" → "gadm" → "osm" อัตโนมัติ
TAMBON_SHP = ""                 # path ไฟล์ขอบเขตตำบลใน Drive เช่น "GIS/tambon_saraburi.shp" (.shp/.gpkg/.zip/.geojson) — เว้นว่าง = ค้นหาอัตโนมัติ
FIRMS_MAP_KEY = ""                # free key: https://firms.modaps.eosdis.nasa.gov/api/map_key (or Colab secret FIRMS_MAP_KEY)
OUTCOME = "total"                                 # "total" หรือชื่อโรค เช่น "COPD", "Pneumonia", "Asthma"
EXPOSURE = "industry_pct"                         # industry_pct | road_density | fire_density
HDC_FALLBACK_CSV = repo_file("hdc_saraburi_pm25_disease_tambon_annual_2565_2570.csv") or ""                             # (ทางเลือก) path ไฟล์ CSV จาก สคริปต์PYดึงข้อมูลผู้ป่วยรายตำบลจากHDC.py (Drive: 3-ColabProjects/Training) ใน Drive
CONF = 0.95
SAVE_TO_DRIVE = False

In [ ]:
import importlib, importlib.util, subprocess, sys
need = {"esda": "esda", "libpysal": "libpysal", "rasterio": "rasterio", "mapclassify": "mapclassify", "osmnx": "osmnx"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, io, json, math, time, zipfile, warnings, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

UA = {"User-Agent": "Mozilla/5.0 (academic research notebook; PM2.5 Saraburi)"}

def setup_thai_font():
    """ให้กราฟ matplotlib แสดงภาษาไทยได้ (ฟอนต์เริ่มต้นไม่มีตัวอักษรไทย → แสดงเป็น □□□)"""
    import matplotlib.font_manager as fm, glob
    base = "/content" if os.path.isdir("/content") else "."
    path = os.path.join(base, "Sarabun-Regular.ttf")
    if not os.path.exists(path):
        try:
            r = requests.get("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", timeout=60)
            if r.status_code == 200 and len(r.content) > 50000: open(path, "wb").write(r.content)
        except Exception:
            pass
    if not os.path.exists(path):                                  # สำรอง: ฟอนต์ไทยจาก apt (TLWG)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1", shell=True)
        cand = sorted(glob.glob("/usr/share/fonts/**/tlwg/Loma.*tf", recursive=True)) + \
               sorted(glob.glob("/usr/share/fonts/**/Garuda.*tf", recursive=True))
        path = cand[0] if cand else None
    if path and os.path.exists(path):
        fm.fontManager.addfont(path)
        name = fm.FontProperties(fname=path).get_name()
        plt.rcParams["font.family"] = ["DejaVu Sans", name]      # ละติน = DejaVu, ไทย = fallback ไปฟอนต์ไทย
        print(f"✓ ฟอนต์ไทยสำหรับกราฟ: {name}")
    else:
        print("⚠ ติดตั้งฟอนต์ไทยไม่ได้ — ข้อความภาษาไทยในกราฟจะแสดงเป็นสี่เหลี่ยม")
setup_thai_font()

# เว็บราชการบางแห่ง (เช่น Air4Thai) ส่งใบรับรอง SSL ไม่ครบ chain → Python ตรวจสอบไม่ผ่าน (SSLError)
# อนุญาตให้ข้ามการตรวจใบรับรองเฉพาะโดเมนเหล่านี้ (ข้อมูลสาธารณะ อ่านอย่างเดียว)
SSL_RELAX_HOSTS = ("air4thai.pcd.go.th", "air4thai.com", "www.air4thai.com")

def http_get(url, timeout=(15, 120), tries=3, quiet=False, **kw):
    """GET พร้อม retry — คืน Response เมื่อ HTTP 200 ไม่เช่นนั้น raise"""
    last = None
    host = url.split("/")[2] if "://" in url else ""
    verify = kw.pop("verify", True)
    for k in range(tries):
        try:
            try:
                r = requests.get(url, headers=UA, timeout=timeout, verify=verify, **kw)
            except requests.exceptions.SSLError:
                if host not in SSL_RELAX_HOSTS or verify is False:
                    raise
                import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
                if not quiet: print(f"   (ใบรับรอง SSL ของ {host} ไม่สมบูรณ์ — เชื่อมต่อโดยไม่ตรวจใบรับรอง)")
                verify = False
                r = requests.get(url, headers=UA, timeout=timeout, verify=False, **kw)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
            if r.status_code == 429 and k < tries - 1:     # ขอถี่เกินโควตา (เช่น Open-Meteo) → รอนานขึ้นแล้วลองใหม่
                wait = int(r.headers.get("Retry-After", 0) or 0) or 30 * (k + 1)
                print(f"   HTTP 429 จาก {host} — รอ {wait} s แล้วลองใหม่", flush=True)
                time.sleep(wait); continue
        except Exception as e:
            last = type(e).__name__ + ": " + str(e)[:100]
        if k < tries - 1:
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"{last}  <- {url[:110]}")

def month_chunks(start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out, cur = [], s
    while cur <= e:
        nxt = min(cur + pd.offsets.MonthEnd(0), e)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt + pd.Timedelta(days=1)
    return out

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# ---- ที่เก็บผลลัพธ์ (ถ้า SAVE_TO_DRIVE=True จะเก็บใน Google Drive เพื่อให้ Notebook อื่นใช้ต่อได้) ----
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTDIR = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
    except Exception as e:
        print("⚠ mount Drive ไม่ได้ จะเก็บไว้ใน /content แทน:", e)
        OUTDIR = "/content/PM25_Saraburi_notebooks"
else:
    OUTDIR = "/content/PM25_Saraburi_notebooks" if os.path.isdir("/content") else "./PM25_Saraburi_notebooks"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR =", OUTDIR)

## 1. ดึงข้อมูลจริง
### 1.1 ขอบเขตตำบล ประชากร จุดความร้อน

In [ ]:
# =====================================================================
#  เตรียมข้อมูลระดับตำบล (ดึงจากอินเทอร์เน็ตทั้งหมด)
#  1) ขอบเขตตำบล  : shapefile ใน Google Drive → OCHA COD-AB Thailand (มีรหัสตำบล 6 หลักตรงกับ HDC) → สำรอง GADM 4.1 → OpenStreetMap
#  2) ประชากร      : WorldPop 2020 (1 km, ปรับรวมรายตำบลด้วย zonal sum)
#  3) จุดความร้อน  : NASA FIRMS (country yearly summary; สำรอง: API + MAP_KEY)
# =====================================================================
import geopandas as gpd
from shapely.geometry import Point
UTM = "EPSG:32647"

COD_BASE = "https://codgis.itos.uga.edu/arcgis/rest/services/COD_External/THA_pcode/FeatureServer"
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_THA_3.json.zip"

def load_tambons_cod():
    """OCHA COD-AB Thailand (ที่มา: กรมแผนที่ทหาร/RTSD) ผ่าน live service ของ ITOS — มีรหัสตำบล (pcode) และชื่อภาษาไทย"""
    import re
    from urllib.parse import quote
    meta = http_get(COD_BASE + "?f=json", timeout=(15, 60)).json()
    layers = meta.get("layers", [])
    lyr = next((l for l in layers if re.search(r"(adm(in)?\s*_?3|tambon|sub.?district)", l["name"], re.I)), None)
    if lyr is None:
        raise RuntimeError(f"ไม่พบชั้นข้อมูลระดับ 3: {[l['name'] for l in layers]}")
    for where in [f"ADM1_EN = '{PROVINCE_EN}'", "ADM1_PCODE = 'TH19'", f"ADM1_EN LIKE '%{PROVINCE_EN}%'"]:
        url = f"{COD_BASE}/{lyr['id']}/query?where={quote(where)}&outFields=*&outSR=4326&f=geojson"
        try:
            g = gpd.read_file(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content))
        except Exception:
            continue
        if len(g):
            break
    else:
        raise RuntimeError("query ไม่ได้ข้อมูล")
    g = g.set_crs("EPSG:4326", allow_override=True)
    namecol = next(c for c in ["ADM3_TH", "ADM3_EN", "admin3Name_th", "admin3Name_en"] if c in g)
    pc = next(c for c in ["ADM3_PCODE", "admin3Pcode"] if c in g)
    g["tambon_id"] = g[pc]; g["name"] = g[namecol]
    g["a_code"] = g[pc].astype(str).str.replace(r"\D", "", regex=True).str[:6]     # TH190101 → 190101 (ตรงกับ HDC)
    if "ADM3_EN" in g: g["name_en"] = g["ADM3_EN"]
    return g[["tambon_id", "a_code", "name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]], \
           f"OCHA COD-AB Thailand (ITOS live service, layer '{lyr['name']}')"

def load_tambons_gadm():
    """GADM 4.1 ระดับ 3 (ตำบล) — ไฟล์ทั้งประเทศ ~30–60 MB"""
    print("   ดาวน์โหลด GADM 4.1 THA level 3 ...")
    raw = http_get(GADM_URL, timeout=(15, 900)).content
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = next(n for n in z.namelist() if n.endswith(".json"))
        g = gpd.read_file(io.BytesIO(z.read(name)))
    g = g[g["NAME_1"].str.replace(" ", "").str.contains(PROVINCE_EN, case=False, na=False)].copy()
    g["tambon_id"] = g["GID_3"]
    g["name"] = g["NL_NAME_3"].where(g.get("NL_NAME_3", pd.Series(dtype=str)).notna() & (g.get("NL_NAME_3", "") != "NA"), g["NAME_3"]) \
        if "NL_NAME_3" in g else g["NAME_3"]
    g["name_en"] = g["NAME_3"]
    cols = ["tambon_id", "name", "name_en", "geometry"]
    if "CC_3" in g and g["CC_3"].astype(str).str.fullmatch(r"\d{6}").mean() > 0.8:
        g["a_code"] = g["CC_3"].astype(str); cols.insert(1, "a_code")
    return g[cols].to_crs("EPSG:4326"), "GADM 4.1 (THA level 3)"

def load_tambons_osm():
    import osmnx as ox
    ox.settings.timeout = 600
    prov = ox.geocode_to_gdf(f"{PROVINCE_EN} Province, Thailand")
    # หมายเหตุ: osmnx รวมแท็กแบบ OR → ถ้าใส่ทั้ง boundary และ admin_level จะได้ขอบเขตทุกระดับ (จังหวัด/อำเภอ/ตำบล) ปนกัน
    g = ox.features_from_polygon(prov.geometry.iloc[0], {"admin_level": "8"}).reset_index()
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if "boundary" in g: g = g[g["boundary"].astype(str) == "administrative"]
    if "admin_level" in g: g = g[g["admin_level"].astype(str) == "8"]
    if "element" in g and (g["element"] == "relation").any(): g = g[g["element"] == "relation"]
    g = g[g.representative_point().within(prov.geometry.iloc[0])]
    namecol = next(c for c in ["name:th", "name", "name:en"] if c in g)
    g["tambon_id"] = g.get("id", g.index).astype(str); g["name"] = g[namecol]
    if "name:en" in g: g["name_en"] = g["name:en"]
    return g[["tambon_id", "name", "geometry"] + (["name_en"] if "name_en" in g else [])], "OpenStreetMap admin_level=8"

# ---------- ขอบเขตตำบลจาก shapefile ใน Google Drive ----------
TAMBON_SEARCH_DIRS = ["/content/drive/MyDrive/PM25_Saraburi_notebooks/boundary",   # ที่แนะนำ
                      "/content/drive/MyDrive/PM25_Saraburi_notebooks",
                      "/content/drive/MyDrive"]                                    # ค้นหาลึกสุด 4 ชั้น
TAMBON_NAME_PAT = r"(tambon|sub.?district|subdist|adm3|admin3|ตำบล|tb_?|_tam)"
SARA_BBOX = (100.55, 14.25, 101.40, 15.05)
MISSING_PARTS = []
TAMBON_CACHE = "saraburi_tambon_boundary.gpkg"   # สำเนาเฉพาะสระบุรี (เล็ก อ่านเร็ว) สร้างอัตโนมัติข้างไฟล์ต้นฉบับ

def find_tambon_files():
    """คืนรายการ path ของ shapefile/gpkg/geojson/zip ที่อาจเป็นขอบเขตตำบลใน Drive (TAMBON_SHP ก่อน แล้วค้นหาอัตโนมัติ)"""
    import re, glob
    MISSING_PARTS.clear()
    if repo_file(TAMBON_CACHE): return [repo_file(TAMBON_CACHE)]          # repository: data/processed/
    if not os.path.isdir("/content/drive/MyDrive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"mount Drive ไม่ได้: {str(e)[:80]}")
    p = (globals().get("TAMBON_SHP") or "").strip()
    if p:
        if not p.startswith("/"): p = os.path.join("/content/drive/MyDrive", p)
        if os.path.exists(p): return [p]
        raise RuntimeError(f"ไม่พบไฟล์ TAMBON_SHP = {p}")
    exts = (".shp", ".gpkg", ".geojson", ".json", ".zip"); found = []
    for root0 in TAMBON_SEARCH_DIRS:
        if not os.path.isdir(root0): continue
        hits, base_depth = [], root0.rstrip("/").count("/")
        for root, dirs, files in os.walk(root0):
            if root.count("/") - base_depth >= 4: dirs[:] = []
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("SatPM25_cache", "ERA5_cache", "WorldPop_cache", "GDAS", "PCD_PM25")]
            for fn in files:
                if fn.lower().endswith(".shx") and re.search(TAMBON_NAME_PAT, fn, re.I) and \
                        (fn[:-4] + ".shp").lower() not in {f.lower() for f in files}:
                    MISSING_PARTS.append(os.path.join(root, fn[:-4] + ".shp"))         # มี .shx แต่ไม่มี .shp
                anyname = root.rstrip("/").endswith("/boundary")          # ในโฟลเดอร์ boundary รับทุกชื่อไฟล์
                if fn.lower().endswith(exts) and (anyname or re.search(TAMBON_NAME_PAT, fn, re.I)) \
                        and not re.search(r"(rates|gistar|lisa|roadside|summary|result)", fn, re.I):
                    if fn.lower().endswith(".shp"):
                        stem = fn[:-4]; have = {f.lower() for f in files}
                        if not {stem.lower() + ".dbf", stem.lower() + ".shx"} <= have:
                            MISSING_PARTS.append(os.path.join(root, fn)); continue      # .dbf/.shx ยังไม่ครบ (อัปโหลดไม่เสร็จ?)
                    hits.append(os.path.join(root, fn))
            if root0 == "/content/drive/MyDrive" and len(hits) > 50: break
        if hits:
            order = {".shp": 0, ".gpkg": 1, ".zip": 2, ".geojson": 3, ".json": 4}
            rank = lambda h: 0 if re.search(TAMBON_NAME_PAT, os.path.basename(h), re.I) else \
                             2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|_ln|_pt|line|point)", os.path.basename(h), re.I) else 1
            hits.sort(key=lambda h: (rank(h), order[os.path.splitext(h)[1].lower()], len(h)))
            found += [h for h in hits if h not in found]
            if any(os.path.basename(h) == TAMBON_CACHE for h in found): break      # เจอสำเนาแล้ว ไม่ต้องค้นต่อ
    found.sort(key=lambda h: os.path.basename(h) != TAMBON_CACHE)                # สำเนาเฉพาะสระบุรีมาก่อน
    if found: return found
    extra = f" (พบ .shp ที่ไฟล์ประกอบไม่ครบ: {[os.path.basename(m) for m in MISSING_PARTS[:3]]})" if MISSING_PARTS else ""
    raise RuntimeError("ไม่พบ shapefile ตำบลใน Drive" + extra + " — วางไฟล์ไว้ที่ MyDrive/PM25_Saraburi_notebooks/boundary/ "
                       "(ชื่อไฟล์มีคำว่า tambon/subdistrict/ตำบล) หรือกำหนด TAMBON_SHP")

def read_tambon_file(path):
    """อ่านไฟล์ขอบเขต + แก้ภาษาไทยเพี้ยน (TIS-620) + กำหนด CRS + ตัดเฉพาะสระบุรี + หาคอลัมน์รหัส/ชื่อตำบล"""
    import re
    src, layer = path, None
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            inner = [n for n in z.namelist() if n.lower().endswith((".shp", ".gpkg", ".geojson", ".json"))]
        if not inner: raise RuntimeError(f"ใน {os.path.basename(path)} ไม่มี .shp/.gpkg/.geojson")
        lvl = lambda n: (0 if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I) else
                         2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|line|pt|point)", n, re.I) else 1)
        inner.sort(key=lambda n: (lvl(n), len(n)))
        src = f"zip://{path}!{inner[0]}"
    if src.lower().endswith(".gpkg"):
        try:
            import pyogrio
            names = [l[0] for l in pyogrio.list_layers(src)]
        except Exception:
            names = []
        if len(names) > 1:
            pref = [n for n in names if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I)]
            layer = pref[0] if pref else names[-1]
    kw = {"layer": layer} if layer else {}
    g = None
    if os.path.getsize(path) > 50e6:                    # ไฟล์ทั้งประเทศขนาดใหญ่ → อ่านเฉพาะกรอบสระบุรี (เร็วกว่ามาก)
        try:
            g = gpd.read_file(src, bbox=(SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3), **kw)
            if len(g) < MIN_TAMBONS: g = None           # พิกัดในไฟล์ไม่ใช่ lat/lon → อ่านทั้งไฟล์
            else: kw["bbox"] = (SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3)
        except Exception:
            g = None
    if g is None: g = gpd.read_file(src, **kw)
    thai = lambda s: bool(re.search(r"[฀-๿]", str(s)))
    garbled = lambda s: bool(re.search(r"[À-ÿ]{2,}", str(s)))
    is_txt = lambda c: c != "geometry" and not pd.api.types.is_numeric_dtype(g[c])
    obj = [c for c in g.columns if is_txt(c)]
    if obj and not any(g[c].astype(str).map(thai).any() for c in obj) and any(g[c].astype(str).map(garbled).any() for c in obj):
        for enc in ("cp874", "tis-620"):
            try:
                g2 = gpd.read_file(src, encoding=enc, **kw)
                if any(g2[c].astype(str).map(thai).any() for c in obj if c in g2): g = g2; obj = [c for c in g.columns if is_txt(c)]; print(f"   อ่านภาษาไทยด้วย encoding={enc}"); break
            except Exception: pass
    g = g[g.geometry.notna() & g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if g.empty: raise RuntimeError("ไม่มี polygon (อาจเป็นไฟล์จุด/เส้น)")
    if g.crs is None:
        x0, y0, x1, y1 = g.total_bounds
        if -180 <= x0 <= 180 and -90 <= y0 <= 90: g = g.set_crs("EPSG:4326")
        else: g = g.set_crs(UTM); print("   ⚠ ไฟล์ไม่มี .prj — สมมติเป็น UTM 47N (EPSG:32647)")
    g = g.to_crs("EPSG:4326")
    up = {c: c.upper() for c in g.columns}
    digits = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    # ---- รหัสตำบล 6 หลัก ----
    code = None
    for c in g.columns:
        if c == "geometry": continue
        d = digits(g[c])
        if (d.str.len() == 6).mean() > 0.8 and d.str[:2].nunique() <= 80: code = d.where(d.str.len() == 6); break
    if code is None:
        pc = next((c for c in g.columns if re.fullmatch(r"(P|PV|PROV|PRO|CHANGWAT|ADM1)_?(CODE|ID|CD)", up[c])), None)
        ac = next((c for c in g.columns if re.fullmatch(r"(A|AP|AMP|AMPHOE|DIST|ADM2)_?(CODE|ID|CD)", up[c])), None)
        tc = next((c for c in g.columns if re.fullmatch(r"(T|TB|TAM|TAMBON|SUBDIST|ADM3)_?(CODE|ID|CD)", up[c])), None)
        if pc and ac and tc:
            p, a, t = digits(g[pc]).str[-2:].str.zfill(2), digits(g[ac]).str[-2:].str.zfill(2), digits(g[tc]).str[-2:].str.zfill(2)
            code = p + a + t
    # ---- ตัดเฉพาะจังหวัดสระบุรี ----
    n_all = len(g); keep = None
    if code is not None and (code.str[:2] == "19").sum() >= MIN_TAMBONS: keep = code.str[:2] == "19"
    if keep is None:
        for c in g.columns:
            if is_txt(c):
                k = g[c].astype(str).str.contains(r"สระบุรี|Saraburi", case=False, na=False)
                if MIN_TAMBONS <= k.sum() < len(g): keep = k; break
    if keep is None and n_all > 140:
        rp = g.representative_point(); w, s, e, n = SARA_BBOX
        keep = rp.x.between(w, e) & rp.y.between(s, n)
        print("   ⚠ ไม่พบคอลัมน์จังหวัด — ตัดด้วยกรอบพิกัดโดยประมาณ (ตรวจตำบลริมขอบจังหวัดอีกครั้ง)")
    if keep is not None:
        g = g[keep.values].copy(); code = code[keep.values] if code is not None else None
    # ---- ชื่อตำบล ----
    pick = lambda pats: next((c for p in pats for c in g.columns if re.fullmatch(p, up[c])), None)
    nc = pick([r"TAM_?NAMT", r"T_?NAME_?T", r"TB_?TN", r"TAMBON_?T(H)?", r"ADM3_?TH", r"NAME_?TH", r"TAM_?NAME", r"T_?NAME", r"TAMBON", r"ADM3_?EN", r"NAME_?3", r"NAME"])
    if nc is None:
        nc = next((c for c in g.columns if is_txt(c) and g[c].astype(str).map(thai).mean() > 0.8), None)
    ec = pick([r"TAM_?NAME_?E", r"TAM_?NAMEE", r"T_?NAME_?E", r"TB_?EN", r"ADM3_?EN", r"NAME_?EN", r"NAME_?3"])
    g["name"] = g[nc].astype(str).str.replace(r"^(ตำบล|ต\.)\s*", "", regex=True) if nc else [f"T{i+1:03d}" for i in range(len(g))]
    if ec and ec != nc: g["name_en"] = g[ec].astype(str)
    if code is not None: g["a_code"] = code.values
    g["tambon_id"] = g["a_code"] if "a_code" in g else g["name"]
    cols = ["tambon_id"] + (["a_code"] if "a_code" in g else []) + ["name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]
    info = f"รหัส={'มี' if 'a_code' in g else 'ไม่มี'}, ชื่อ={nc}, ตัดจาก {n_all} → {len(g)} หน่วย"
    return g[cols].reset_index(drop=True), info

def load_tambons_drive():
    import time
    errs = []; t0 = time.time()
    for path in find_tambon_files()[:6]:
        short = path.replace("/content/drive/MyDrive/", "MyDrive/")
        try:
            g, info = read_tambon_file(path)
            if len(g) < MIN_TAMBONS: raise RuntimeError(f"ได้ {len(g)} หน่วย ({info})")
            print(f"   ไฟล์: {short} ({info}; {time.time() - t0:.0f} s)")
            if os.path.basename(path) != TAMBON_CACHE:
                try:                                      # ครั้งต่อไปอ่านสำเนานี้แทน (ไม่กี่วินาที)
                    cp = os.path.join(os.path.dirname(path), TAMBON_CACHE)
                    g.to_file(cp, driver="GPKG"); print(f"   บันทึกสำเนาเฉพาะสระบุรี → {cp.replace('/content/drive/MyDrive/', 'MyDrive/')}")
                except Exception as e:
                    print(f"   (บันทึกสำเนาไม่ได้: {str(e)[:60]})")
            return g, f"Drive: {os.path.basename(path)}"
        except Exception as e:
            errs.append(f"{os.path.basename(path)}: {str(e)[:70]}")
    raise RuntimeError("ไฟล์ใน Drive ใช้ไม่ได้ — " + " | ".join(errs))

MIN_TAMBONS = 60     # สระบุรีมี ~108–111 ตำบล ถ้าได้น้อยกว่านี้แสดงว่าได้ขอบเขตผิดระดับ (เช่น อำเภอ)

def clean_units(g):
    """ตัด polygon ซ้ำ และ polygon 'แม่' ที่ครอบหน่วยอื่น ≥ 2 หน่วย (เช่น อำเภอ/จังหวัดที่ปนมา)"""
    n0 = len(g)
    g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["geometry"] = g.geometry.buffer(0)
    key = g.to_crs(UTM).geometry.apply(lambda x: (round(x.area / 1e4), round(x.centroid.x / 50), round(x.centroid.y / 50)))
    g = g[~key.duplicated()].reset_index(drop=True)
    rp = g.representative_point(); sidx = rp.sindex
    area = g.to_crs(UTM).area.values
    parent = []
    for i, geom in enumerate(g.geometry):
        inside = [j for j in sidx.query(geom, predicate="contains") if j != i and area[j] < 0.8 * area[i]]
        parent.append(len(inside) >= 2)               # ครอบหน่วยที่เล็กกว่าอย่างน้อย 2 หน่วย = ขอบเขตระดับบน
    g = g[~np.array(parent)].reset_index(drop=True)
    if "name" in g:
        g = g[~g["name"].astype(str).str.match(r"^(อำเภอ|จังหวัด|เขต)")].reset_index(drop=True)
    if len(g) != n0:
        print(f"   ทำความสะอาดขอบเขต: {n0} → {len(g)} หน่วย (ตัด polygon ซ้ำ/ระดับอำเภอ-จังหวัดที่ปนมา)")
    return g

KV_URL = "https://raw.githubusercontent.com/kongvut/thai-province-data/master/api/latest/sub_district.json"

def add_tambon_codes(g):
    """เติมรหัสตำบล 6 หลัก (มท.) ให้ขอบเขตที่ไม่มีรหัส (GADM/OSM): จับคู่ตำแหน่งตำบลก่อน แล้วจับคู่ชื่อ"""
    import re
    js = http_get(KV_URL, timeout=(15, 120)).json()
    kv = pd.DataFrame([{"a_code": str(t["id"]), "th": (t["name"] or {}).get("th", ""), "en": (t["name"] or {}).get("en", ""),
                        "lat": t.get("lat"), "lon": t.get("long")} for t in js if t.get("id")])
    pts = kv.dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs="EPSG:4326")
    j = gpd.sjoin(pts, g[["geometry"]].reset_index().rename(columns={"index": "gi"}), predicate="within")
    prov = j.a_code.str[:2].mode().iloc[0]
    kv = kv[kv.a_code.str[:2] == prov]
    norm_th = lambda x: re.sub(r"^(ตำบล|ต\.)", "", str(x)).replace(" ", "").strip()
    norm_en = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
    by_th = dict(zip(kv.th.map(norm_th), kv.a_code)); by_en = dict(zip(kv.en.map(norm_en), kv.a_code))
    code = pd.Series(index=g.index, dtype=object)
    for col, table, f in [("name", by_th, norm_th), ("name_en", by_en, norm_en), ("name", by_en, norm_en)]:
        if col in g:
            m = g[col].map(f).map(table); code = code.fillna(m)
    jj = j[j.a_code.str[:2] == prov].drop_duplicates("gi").set_index("gi")["a_code"]
    code = code.fillna(jj.reindex(g.index))
    g["a_code"] = code
    th = dict(zip(kv.a_code, kv.th)); g["name_th"] = g["a_code"].map(th)
    if g["name_th"].notna().mean() > 0.8: g["name"] = g["name_th"].fillna(g["name"])
    print(f"   เติมรหัสตำบล (มท.) ได้ {g.a_code.notna().sum()} / {len(g)} หน่วย (จังหวัดรหัส {prov})")
    return g.drop(columns="name_th")

def load_tambons():
    loaders = {"drive": load_tambons_drive, "cod": load_tambons_cod, "gadm": load_tambons_gadm, "osm": load_tambons_osm}
    first = BOUNDARY_SOURCE if BOUNDARY_SOURCE in loaders else "drive"
    order = [loaders[first]] + [f for k, f in loaders.items() if k != first]
    for f in order:
        try:
            g, src = f(); g = clean_units(g.reset_index(drop=True))
            if len(g) < MIN_TAMBONS:
                raise RuntimeError(f"ได้เพียง {len(g)} หน่วย — น่าจะไม่ใช่ระดับตำบล")
            if "a_code" not in g or g["a_code"].isna().mean() > 0.2:
                try:
                    g = add_tambon_codes(g)
                except Exception as e:
                    print(f"   ⚠ เติมรหัสตำบลไม่ได้: {str(e)[:80]} — การจับคู่ข้อมูล HDC จะใช้ชื่อแทน")
            print(f"✓ ขอบเขตตำบล: {len(g)} ตำบล  ({src})"); return g, src
        except Exception as e:
            print(f"   ✗ {f.__name__}: {str(e)[:120]}")
    raise RuntimeError("โหลดขอบเขตตำบลไม่สำเร็จ")

WORLDPOP_URLS = [
    "https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/2020/THA/tha_ppp_2020_1km_Aggregated_UNadj.tif",  # ~3 MB
    "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/THA/tha_ppp_2020_UNadj.tif",                            # ~250 MB (100 m)
]

def add_population(g):
    """ดาวน์โหลด GeoTIFF ของ WorldPop ลงเครื่อง แล้วรวมประชากรรายตำบล (zonal sum)"""
    import rasterio
    from rasterio.mask import mask
    for u in [WORLDPOP_URLS[0]] + WORLDPOP_URLS:          # ไฟล์ 1 km ลอง 2 ครั้ง (server ตัดการเชื่อมต่อบ่อย) แล้วจึงใช้ 100 m
        fn = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
        try:
            if not (os.path.exists(fn) and os.path.getsize(fn) > 1e6):
                print(f"   ดาวน์โหลด {u.split('/')[-1]} ...")
                with requests.get(u, headers=UA, stream=True, timeout=(15, 600)) as r:
                    r.raise_for_status()
                    with open(fn, "wb") as f:
                        for chunk in r.iter_content(1 << 20): f.write(chunk)
            with rasterio.open(fn) as src:
                gg = g.to_crs(src.crs); pop = []
                for geom in gg.geometry:
                    arr, _ = mask(src, [geom], crop=True, all_touched=False, filled=True, nodata=0)
                    a = arr[0].astype(float); a[~np.isfinite(a) | (a < 0)] = 0; pop.append(a.sum())
            if sum(pop) <= 0:
                raise RuntimeError("ผลรวมประชากร = 0")
            g["pop"] = np.round(pop); print(f"✓ ประชากร WorldPop 2020: รวม {g['pop'].sum():,.0f} คน  ({u.split('/')[-1]})")
            return g, "WorldPop 2020 UN-adjusted (" + u.split("/")[-1] + ")"
        except Exception as e:
            print(f"   ✗ WorldPop {u.split('/')[-1]}: {type(e).__name__}: {str(e)[:100]}")
            if os.path.exists(fn) and os.path.getsize(fn) < 1e6: os.remove(fn)
    print("⚠ ไม่ได้ข้อมูลประชากร — ใช้พื้นที่แทนเป็นตัวหารสำรอง"); g["pop"] = np.nan
    return g, None

FIRMS_PRODUCTS = ["viirs-snpp", "viirs-noaa20", "modis"]       # ลำดับความชอบ (VIIRS 375 m ละเอียดกว่า MODIS 1 km)
FIRMS_API_SRC = {"viirs-snpp": ["VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"], "viirs-noaa20": ["VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"],
                 "modis": ["MODIS_SP", "MODIS_NRT"]}

def _firms_key():
    if FIRMS_MAP_KEY: return FIRMS_MAP_KEY
    try:
        from google.colab import userdata; return userdata.get("FIRMS_MAP_KEY")
    except Exception:
        return None

FIRMS_GAPS = []        # ช่วงวันที่ API ไม่มีข้อมูล (ไม่ใช่ "ไม่มีไฟ") — ใช้ตัดออก/ปรับตัวหาร

def firms_api_year(key, sources, y, bbox):
    """FIRMS area API ทีละ 5 วัน: ใช้ SP (standard, ตรวจสอบแล้ว) ก่อน แล้วต่อด้วย NRT เฉพาะวันหลังวันสุดท้ายของ SP
    (SP ออกช้ากว่าจริง ~2–3 เดือน ; NRT เก็บย้อนหลังแค่ ~2 เดือน → ต้องใช้ทั้งสองต่อกันจึงครบปีปัจจุบัน)"""
    end = min(pd.Timestamp(f"{y}-12-31"), pd.Timestamp.now().normalize())
    got, how, last = [], [], None
    for src in sources:
        out, bad = [], 0
        start = pd.Timestamp(f"{y}-01-01") if last is None else last + pd.Timedelta(days=1)
        for d in pd.date_range(start, end, freq="5D"):
            u = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{src}/"
                 f"{bbox[0]:.3f},{bbox[1]:.3f},{bbox[2]:.3f},{bbox[3]:.3f}/5/{d:%Y-%m-%d}")
            try:
                txt = http_get(u, timeout=(15, 120), tries=3).text
                if "latitude" not in txt[:300]:
                    bad += 1
                    if bad == 1: print(f"   FIRMS API {src}: {txt[:120].strip()}")
                    if "invalid" in txt.lower() or "key" in txt.lower(): break
                    continue
                t = pd.read_csv(io.StringIO(txt))
                if len(t): out.append(t[pd.to_datetime(t.acq_date) <= end])
            except Exception as e:
                bad += 1
        if out:
            t = pd.concat(out, ignore_index=True)
            t = t[pd.to_datetime(t.acq_date) >= start]
            if len(t):
                lo, hi = pd.to_datetime(t.acq_date).min(), pd.to_datetime(t.acq_date).max()
                if got and (lo - start).days > 2:
                    print(f"   ⚠ FIRMS {y}: ไม่มีข้อมูลช่วง {start:%Y-%m-%d} → {lo - pd.Timedelta(days=1):%Y-%m-%d} (SP ยังไม่ออก/NRT ลบไปแล้ว)")
                    FIRMS_GAPS.append((start, lo - pd.Timedelta(days=1)))
                got.append(t)
                how.append(f"{src} {lo:%d/%m}–{hi:%d/%m}"); last = hi
        if last is not None and last >= end - pd.Timedelta(days=2): break
    if not got: return None, None
    t = pd.concat(got, ignore_index=True).drop_duplicates()
    print(f"   FIRMS {y} ผ่าน API: " + " + ".join(how) + f" (ถึง {pd.to_datetime(t.acq_date).max():%Y-%m-%d})")
    return t, "API " + "+".join(how)

def _firms_year(product, y, bbox):
    try:
        url = f"https://firms.modaps.eosdis.nasa.gov/data/country/{product}/{y}/{product}_{y}_Thailand.csv"
        return pd.read_csv(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content)), "yearly"
    except Exception:
        key = _firms_key()
        if not key: return None, None
        t, how = firms_api_year(key, FIRMS_API_SRC[product], y, bbox)
        return (t, how) if t is not None else (None, None)

def load_fires(g):
    """จุดความร้อน VIIRS S-NPP 375 m รายปี (เครื่องวัดเดียวตลอดช่วง เพื่อให้เทียบข้ามปีได้)
    ปีที่ไม่มีไฟล์รายปี (เช่น ปีปัจจุบัน) → ดึงผ่าน API ถ้ามี FIRMS_MAP_KEY มิฉะนั้นตัดปีนั้นออก (ไม่เติมค่า 0)"""
    global YEARS, YEAR_FRAC
    b = g.total_bounds; bbox = (b[0] - 0.05, b[1] - 0.05, b[2] + 0.05, b[3] + 0.05)
    prod, today = "viirs-snpp", pd.Timestamp.now().normalize()
    frames, how, got = [], [], []
    for y in YEARS:
        if y > today.year: continue
        t, h = _firms_year(prod, y, bbox)
        if t is None:
            print(f"   ✗ FIRMS {prod} {y}: ไม่มีไฟล์รายปี" + ("" if _firms_key() else " — ใส่ FIRMS_MAP_KEY (ฟรี) เพื่อดึงปีนี้ผ่าน API"))
            continue
        t = t[pd.to_numeric(t.longitude, errors="coerce").between(bbox[0], bbox[2]) &
              pd.to_numeric(t.latitude, errors="coerce").between(bbox[1], bbox[3])]
        frames.append(t); how.append(f"{y}:{h}"); got.append(y)
        print(f"   ✓ FIRMS {prod} {y} ({h}): {len(t):,} จุดในกรอบจังหวัด", flush=True)
    if not frames:
        raise RuntimeError("ดึง FIRMS ไม่ได้ — ใส่ FIRMS_MAP_KEY (ขอฟรีที่ https://firms.modaps.eosdis.nasa.gov/api/map_key)")
    missing = [y for y in YEARS if y not in got]
    if missing:
        print(f"⚠ ไม่มีข้อมูลจุดความร้อนปี {missing} — ตัดปีนั้นออกจากการวิเคราะห์")
    YEARS = got
    F = pd.concat(frames, ignore_index=True)
    ok = F["confidence"].astype(str).str.lower().str[0].isin(["n", "h"])
    if "type" in F:                                   # type 0 = presumed vegetation fire (ตัดแหล่งความร้อนถาวร เช่น เตาเผาปูน)
        ok &= pd.to_numeric(F["type"], errors="coerce").fillna(0).eq(0)
    F = F[ok].copy()
    F["time_local"] = pd.to_datetime(F["acq_date"].astype(str) + " " + F["acq_time"].astype(int).astype(str).str.zfill(4),
                                     format="%Y-%m-%d %H%M") + pd.Timedelta(hours=7)
    F = F[F.time_local.dt.year.isin(YEARS)]
    pts = gpd.GeoDataFrame(F, geometry=gpd.points_from_xy(F.longitude, F.latitude), crs="EPSG:4326")
    pts = gpd.sjoin(pts, g[["geometry"]], predicate="within", how="inner").rename(columns={"index_right": "tidx"})
    pts["year"] = pts.time_local.dt.year; pts["month"] = pts.time_local.dt.to_period("M")
    YEAR_FRAC = {y: 1.0 for y in YEARS}               # ปีที่ยังไม่ครบ (ปีปัจจุบัน) นับตามสัดส่วนวันที่มีข้อมูล
    if today.year in YEARS:
        gap_days = sum((e - s).days + 1 for s, e in FIRMS_GAPS if s.year == today.year)
        YEAR_FRAC[today.year] = ((pts.time_local.max().normalize() - pd.Timestamp(today.year, 1, 1)).days + 1 - gap_days) / 365.0
    print(f"✓ FIRMS {prod}: {len(pts):,} จุดความร้อนในจังหวัด ปี {YEARS[0]}–{YEARS[-1]}")
    return pts, f"NASA FIRMS {prod} ({', '.join(how)})"

TAMBON, BND_SRC = load_tambons()
TAMBON["area_km2"] = TAMBON.to_crs(UTM).area / 1e6
TAMBON, POP_SRC = add_population(TAMBON)
FIRES, FIRE_SRC = load_fires(TAMBON)
for y in YEARS:
    TAMBON[f"fires_{y}"] = np.bincount(FIRES.loc[FIRES.year == y, "tidx"], minlength=len(TAMBON))
TAMBON["fires_all"] = TAMBON[[f"fires_{y}" for y in YEARS]].sum(axis=1)
TAMBON["density_km2"] = TAMBON.fires_all / TAMBON.area_km2 / sum(YEAR_FRAC.values())    # จุด/km²/ปี
DATA_SOURCES = {"boundaries": BND_SRC, "population": POP_SRC, "fires": FIRE_SRC}
print(pd.Series(DATA_SOURCES))
TAMBON.drop(columns="geometry").describe().round(2)

In [ ]:
if TAMBON["pop"].isna().all():
    raise RuntimeError("ต้องมีข้อมูลประชากรเป็นตัวหารของอัตรา — WorldPop ดาวน์โหลดไม่สำเร็จ ลองรันเซลล์ก่อนหน้าใหม่")
if "a_code" not in TAMBON:
    print("⚠ ขอบเขตไม่มีรหัสตำบล — จะจับคู่ข้อมูลผู้ป่วยด้วยชื่อตำบล (แนะนำ BOUNDARY_SOURCE='cod')")

### 1.2 ตัวแปรการสัมผัส (exposure) จาก OSM
สัดส่วนพื้นที่อุตสาหกรรม/เหมือง (%) และความหนาแน่นถนนสายหลัก (km/km²) ต่อตำบล

In [ ]:
# ---- เรียก Overpass API โดยตรง (timeout สั้น, ลองหลาย server, แสดงความคืบหน้า) ----
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import polygonize, unary_union
OVERPASS = ["https://overpass-api.de/api/interpreter", "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.private.coffee/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
# overpass-api.de ตอบ HTTP 406 กับ User-Agent บางแบบ → ใช้ UA ที่ระบุตัวตนชัดเจนตามนโยบายการใช้งาน
OP_HEADERS = {"User-Agent": "PM25-Saraburi-research-notebook/1.1 (academic; Google Colab)", "Accept": "*/*"}

GEOFABRIK_URL = "https://download.geofabrik.de/asia/thailand-latest-free.shp.zip"
GF_LAYERS = {"roads": "gis_osm_roads_free_1.shp", "landuse_a": "gis_osm_landuse_a_free_1.shp",
             "pois_a": "gis_osm_pois_a_free_1.shp", "pois": "gis_osm_pois_free_1.shp"}

def _area_bbox(area):
    a = area.strip("()")
    if a.startswith("around:"):
        r_, lat_, lon_ = map(float, a.split(":")[1].split(","))
        dlat = r_ / 111320; dlon = dlat / np.cos(np.deg2rad(lat_))
        return (lon_ - dlon, lat_ - dlat, lon_ + dlon, lat_ + dlat), (lat_, lon_, r_)
    s_, w_, n_, e_ = map(float, a.split(",")); return (w_, s_, e_, n_), None

def geofabrik_gdf(gf, area):
    """ไฟล์ OSM ทั้งประเทศจาก Geofabrik (shapefile อัปเดตทุกวัน) — ดาวน์โหลดครั้งเดียว ~0.5–1 GB แล้วอ่านเฉพาะกรอบพื้นที่"""
    layer, fclasses, col = gf
    fn = ("/content" if os.path.isdir("/content") else ".") + "/thailand-latest-free.shp.zip"
    t0 = time.time()
    try:
        if not (os.path.exists(fn) and os.path.getsize(fn) > 1e8):
            print("   ดาวน์โหลด Geofabrik thailand-latest-free.shp.zip (ครั้งเดียว, ~1–3 นาที) ...", flush=True)
            with requests.get(GEOFABRIK_URL, headers=OP_HEADERS, stream=True, timeout=(15, 600)) as r:
                r.raise_for_status()
                with open(fn + ".part", "wb") as f:
                    for ch in r.iter_content(1 << 22): f.write(ch)
            os.replace(fn + ".part", fn)
        bbox, circ = _area_bbox(area)
        g = gpd.read_file(f"/vsizip/{fn}/{GF_LAYERS[layer]}", bbox=bbox)
        if fclasses: g = g[g["fclass"].isin(fclasses)]
        if circ:
            lat_, lon_, r_ = circ
            d = g.to_crs(UTM).distance(gpd.GeoSeries([Point(lon_, lat_)], crs="EPSG:4326").to_crs(UTM).iloc[0])
            g = g[d <= r_]
        g = g.rename(columns={"fclass": col}).copy()
        print(f"   ✓ Geofabrik {layer}: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g.to_crs("EPSG:4326")
    except Exception as e:
        print(f"   ✗ Geofabrik ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:90]}", flush=True)
        if os.path.exists(fn + ".part"): os.remove(fn + ".part")
        return None

def ohsome_gdf(ohsome_filter, area, kind):
    """ohsome API (HeiGIT) — ดึง OSM ตามตัวกรอง; เสถียรกว่า Overpass ที่มักเต็ม (502 at capacity)"""
    a = area.strip("()")
    if a.startswith("around:"):
        r_, lat_, lon_ = a.split(":")[1].split(","); geo = {"bcircles": f"{lon_},{lat_},{r_}"}
    else:
        s_, w_, n_, e_ = a.split(","); geo = {"bboxes": f"{w_},{s_},{e_},{n_}"}
    gfilt = "geometry:line" if kind == "line" else "(geometry:polygon or geometry:point)"
    t0 = time.time()
    try:
        r = requests.post("https://api.ohsome.org/v1/elements/geometry",
                          data={**geo, "filter": f"({ohsome_filter}) and {gfilt}", "properties": "tags"},
                          headers=OP_HEADERS, timeout=(15, 240))
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:80]}")
        feats = r.json().get("features", [])
        g = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326") if feats else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        print(f"   ✓ ohsome: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g
    except Exception as e:
        print(f"   ✗ ohsome ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:80]}", flush=True)
        return None

def overpass_gdf(selectors, area, kind, timeout_s=90, ohsome_filter=None, cache_key=None, gf=None):
    """ลำดับ: cache ในเครื่อง → Geofabrik (ไฟล์) → ohsome → Overpass หลาย server"""
    import hashlib
    cdir = "/content/osm_cache" if os.path.isdir("/content") else "osm_cache"; os.makedirs(cdir, exist_ok=True)
    cfile = os.path.join(cdir, hashlib.md5((str(selectors) + area + kind).encode()).hexdigest() + ".gpkg")
    if os.path.exists(cfile):
        g = gpd.read_file(cfile); print(f"   ✓ ใช้ข้อมูล OSM ที่เก็บไว้ ({len(g):,} รายการ)"); return g
    g = geofabrik_gdf(gf, area) if gf else None
    if g is None or g.empty:
        g = ohsome_gdf(ohsome_filter, area, kind) if ohsome_filter else None
    if g is None or g.empty:
        g = _overpass_raw(selectors, area, kind, timeout_s)
    if len(g):
        try: g.to_file(cfile, driver="GPKG")
        except Exception: pass
    return g

def _overpass_raw(selectors, area, kind, timeout_s=90):
    # selectors เช่น ['way["landuse"="quarry"]'] ; area = '(s,w,n,e)' หรือ '(around:15000,lat,lon)'
    # kind = 'polygon' (way ปิด + relation + node) หรือ 'line'
    q = f"[out:json][timeout:{timeout_s}];(" + "".join(f"{sel}{area};" for sel in selectors) + ");out geom tags;"
    for url in [u for u in OVERPASS for _ in range(2)]:
        t0 = time.time()
        try:
            r = requests.post(url, data={"data": q}, headers=OP_HEADERS, timeout=(15, timeout_s + 15))
            if r.status_code in (403, 405, 406):                     # บาง server ไม่รับ POST/UA นี้ → ลอง GET
                r = requests.get(url, params={"data": q}, headers=OP_HEADERS, timeout=(15, timeout_s + 15))
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:60].strip()}")
            els = r.json().get("elements", [])
        except Exception as e:
            print(f"   ✗ {url.split('/')[2]} ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:70]}", flush=True)
            time.sleep(10); continue
        rows = []
        for el in els:
            geom = None
            if el["type"] == "way" and "geometry" in el:
                xy = [(p["lon"], p["lat"]) for p in el["geometry"]]
                if kind == "line" and len(xy) >= 2: geom = LineString(xy)
                if kind == "polygon" and len(xy) >= 4 and xy[0] == xy[-1]: geom = Polygon(xy)
            elif el["type"] == "relation" and kind == "polygon":
                lines = [LineString([(p["lon"], p["lat"]) for p in m["geometry"]]) for m in el.get("members", [])
                         if m.get("role", "outer") in ("outer", "") and len(m.get("geometry", [])) >= 2]
                polys = list(polygonize(unary_union(lines))) if lines else []
                if polys: geom = unary_union(polys)
            elif el["type"] == "node" and kind == "polygon" and "lat" in el:
                geom = Point(el["lon"], el["lat"])
            if geom is not None and not geom.is_empty:
                rows.append({**el.get("tags", {}), "osm_type": el["type"], "osm_id": el["id"], "geometry": geom})
        g = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326") if rows else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        print(f"   ✓ {url.split('/')[2]}: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g
    return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

# ---- แหล่งกำเนิดจาก OpenStreetMap รายตำบล ----
PROV_POLY = TAMBON.geometry.union_all() if hasattr(TAMBON.geometry, "union_all") else TAMBON.unary_union
w_, s_, e_, n_ = PROV_POLY.buffer(0.01).bounds; BBOX = f"({s_:.4f},{w_:.4f},{n_:.4f},{e_:.4f})"
TU = TAMBON.to_crs(UTM)
print("OSM: พื้นที่อุตสาหกรรม/เหมือง ...", flush=True)
IND = overpass_gdf(['way["landuse"~"^(industrial|quarry)$"]', 'relation["landuse"~"^(industrial|quarry)$"]',
                    'way["man_made"="works"]', 'relation["man_made"="works"]'], BBOX, "polygon",
                   ohsome_filter="landuse in (industrial, quarry) or man_made=works",
                   gf=("landuse_a", ["industrial", "quarry"], "landuse"))
IND = (IND[IND.geometry.type.isin(["Polygon", "MultiPolygon"])] if len(IND) else IND).to_crs(UTM)
print("OSM: ถนนสายหลัก ...", flush=True)
ROADS = overpass_gdf(['way["highway"~"^(motorway|trunk|primary|secondary)(_link)?$"]'], BBOX, "line",
                     ohsome_filter="highway in (motorway, trunk, primary, secondary, motorway_link, trunk_link, primary_link)",
                     gf=("roads", ["motorway", "trunk", "primary", "secondary", "motorway_link", "trunk_link", "primary_link"],
                         "highway")).to_crs(UTM)

def _union(gs):
    gs = gs.buffer(0); return gs.union_all() if hasattr(gs, "union_all") else gs.unary_union
TAMBON["industry_pct"] = TAMBON["quarry_pct"] = TAMBON["road_density"] = np.nan
if len(IND):
    U = _union(IND.geometry); TAMBON["industry_pct"] = [g.intersection(U).area / g.area * 100 for g in TU.geometry]
    qu = IND[IND["landuse"].astype(str) == "quarry"] if "landuse" in IND else IND.iloc[:0]
    Uq = _union(qu.geometry) if len(qu) else None
    TAMBON["quarry_pct"] = [g.intersection(Uq).area / g.area * 100 if Uq is not None else 0.0 for g in TU.geometry]
if len(ROADS):
    TAMBON["road_density"] = [ROADS.geometry.intersection(g).length.sum() / 1000 / (g.area / 1e6) for g in TU.geometry]
if not len(IND) or not len(ROADS):
    print("⚠ OSM บางชั้นดึงไม่ได้ (server ไม่ว่าง) — ตัวแปรนั้นจะเป็น NaN และถูกข้ามอัตโนมัติ; รันเซลล์นี้ซ้ำภายหลังได้")
TAMBON["fire_density"] = TAMBON["density_km2"]
TAMBON["pop_density"] = TAMBON["pop"] / TAMBON["area_km2"]

TAMBON[["name", "pop", "pop_density", "industry_pct", "quarry_pct", "road_density", "fire_density"]].describe().round(2)

### 1.3 จำนวนผู้ป่วยจาก MOPH Open Data (ข้อมูล HDC)
API ให้ข้อมูลระดับ **สถานบริการ × รหัสพื้นที่ 8 หลัก** → รวมเป็นตำบลจาก 6 หลักแรก (นับเฉพาะผู้ป่วยที่อยู่ในจังหวัด)

In [ ]:
API = "https://opendata.moph.go.th/api/report_data"
COLMAP = {"r1_1": "COPD", "r1_2": "Asthma", "r1_3": "Pneumonia", "r1_4": "Influenza", "r1_5": "Acute pharyngitis",
          "r1_6": "Chronic rhinitis", "r1_7": "Bronchitis", "r2_1": "Ischemic heart diseases",
          "r2_2": "Cerebrovascular diseases (Stroke)", "r3": "กลุ่มโรคตาอักเสบ", "r4": "กลุ่มโรคผิวหนังอักเสบ",
          "r5": "UNKNOWN_r5", "r6": "UNKNOWN_r6"}
DISEASES = [v for v in COLMAP.values() if not v.startswith("UNKNOWN")]

def fetch_opendata(year, province="19", page=10000):
    rows, off = [], 0
    while True:
        for attempt in range(7):                       # 429 = ส่งคำขอถี่เกิน → รอแล้วลองใหม่ (2, 4, 8, ... วินาที)
            r = requests.post(API, json={"tableName": "s_pm25_disease", "year": str(year), "province": province,
                                         "type": "json", "limit": page, "offset": off},
                              headers={"User-Agent": "curl/8"}, timeout=(15, 180))
            if r.status_code in (429, 502, 503, 504):
                wait = int(r.headers.get("Retry-After", 0) or 0) or 2 ** (attempt + 1)
                print(f"   ปี {year}: HTTP {r.status_code} — รอ {wait} s แล้วลองใหม่ ({attempt + 1}/7)", flush=True)
                time.sleep(wait); continue
            break
        if r.status_code not in (200, 201): raise RuntimeError(f"HTTP {r.status_code}")
        js = r.json(); d = js.get("data", [])
        rows += d
        if not d or len(rows) >= int(js.get("total", len(rows))): break
        off += page; time.sleep(0.4)
    return pd.DataFrame(rows)

frames, last_proc = [], {}
try:
    for y in HDC_YEARS:
        try:
            raw = fetch_opendata(y)
        except Exception as e:
            if not frames and y == HDC_YEARS[-1]: raise
            print(f"   ปีงบ {y}: ดึงไม่ได้ ({str(e)[:60]}) — ข้าม"); continue
        if raw.empty or "areacode" not in raw: print(f"   ปีงบ {y}: ไม่มีข้อมูลใน MOPH Open Data"); continue
        raw["a_code"] = raw["areacode"].astype(str).str.zfill(8).str[:6]
        raw = raw[raw.a_code.str[:2] == "19"]
        rc = [c for c in raw.columns if c in COLMAP]
        raw[rc] = raw[rc].apply(pd.to_numeric, errors="coerce").fillna(0)
        g = raw.groupby("a_code")[rc].sum().rename(columns=COLMAP).reset_index(); g["year"] = y
        frames.append(g)
        last_proc[y] = str(pd.to_datetime(raw["date_com"].astype(str), format="%Y%m%d%H%M", errors="coerce").max())
        print(f"✓ ปีงบ {y}: {len(raw):,} แถว → {g.a_code.nunique()} ตำบล (ประมวลผลล่าสุด {last_proc[y]})")
        time.sleep(2)                                  # เว้นระยะระหว่างปี ลดโอกาสโดน rate limit
    if not frames: raise RuntimeError("MOPH Open Data ไม่มีข้อมูลปีที่ขอ")
    HDC = pd.concat(frames, ignore_index=True); HDC_SRC = "MOPH Open Data API (s_pm25_disease)"
    # ตัดปีงบที่ยังประมวลผลไม่ครบ (วันที่ประมวลผลล่าสุดก่อนสิ้นปีงบ 30 ก.ย.)
    inc = [y for y, lp in last_proc.items() if pd.notna(pd.to_datetime(lp, errors="coerce"))
           and pd.to_datetime(lp) < pd.Timestamp(y - 543, 9, 30)]
    if inc:
        print(f"⚠ ตัดปีงบที่ข้อมูลยังไม่ครบ 12 เดือน: {inc}"); HDC = HDC[~HDC.year.isin(inc)]
    print("ปีงบที่ใช้:", sorted(HDC.year.unique()))
except Exception as e:
    print(f"✗ MOPH Open Data: {str(e)[:120]}")
    if not HDC_FALLBACK_CSV:
        raise RuntimeError("เชื่อมต่อ MOPH Open Data จาก Colab ไม่ได้ — รัน สคริปต์PYดึงข้อมูลผู้ป่วยรายตำบลจากHDC.py (Drive: 3-ColabProjects/Training) บนเครื่องในไทย "
                           "แล้วอัปโหลด CSV ไป Drive และใส่ path ใน HDC_FALLBACK_CSV")
    from google.colab import drive; drive.mount("/content/drive")
    HDC = pd.read_csv(HDC_FALLBACK_CSV, encoding="utf-8-sig", dtype={"a_code": str})
    HDC = HDC[HDC.year.isin(HDC_YEARS)]; HDC_SRC = "CSV from สคริปต์PYดึงข้อมูลผู้ป่วยรายตำบลจากHDC.py (Drive: 3-ColabProjects/Training)"
for c in DISEASES:
    if c not in HDC: HDC[c] = 0
HDC["total"] = HDC[DISEASES].sum(axis=1)
print("\nผู้ป่วยรวมรายปี:", HDC.groupby("year")["total"].sum().astype(int).to_dict())

### 1.4 รวมเป็นตาราง ตำบล × ปี และกำหนดกลุ่มการสัมผัส
- **Person-years** = ประชากร (WorldPop 2020) × 1 ปี ต่อแถว
- กลุ่มการสัมผัส 3 ระดับ (`low` = กลุ่มอ้างอิง) — ถ้าตำบลจำนวนมากมีค่า 0 จะใช้ `none / low / high` (แบ่งกลุ่มที่ >0 ด้วยค่ามัธยฐาน)

In [ ]:
key = "a_code" if "a_code" in TAMBON and TAMBON["a_code"].notna().any() else "name"
if key == "name" and "a_name" not in HDC:
    raise RuntimeError("ขอบเขตตำบลไม่มีรหัสตำบล และข้อมูลผู้ป่วยไม่มีชื่อตำบล → จับคู่ไม่ได้ (ดูข้อความในส่วนที่ 1.1)")
HDC[key] = HDC[key if key in HDC else "a_name"].astype(str)
TAMBON[key] = TAMBON[key].astype(str)
D = HDC.merge(TAMBON.drop(columns="geometry")[[key, "name", "pop", "pop_density", "industry_pct", "road_density",
                                               "fire_density", "area_km2"] if key == "a_code" else
                                              [key, "pop", "pop_density", "industry_pct", "road_density", "fire_density", "area_km2"]],
              on=key, how="inner")
D = D[D["pop"] > 0].copy(); D["py"] = D["pop"]
print(f"จับคู่ได้ {D[key].nunique()} ตำบล × {D.year.nunique()} ปี = {len(D)} แถว")

def make_groups(x):
    x = pd.Series(x)
    if (x <= 0).mean() > 1 / 3:
        med = x[x > 0].median()
        return pd.Series(np.select([x <= 0, x <= med], ["0_none", "1_low"], "2_high"), index=x.index)
    return pd.qcut(x.rank(method="first"), 3, labels=["0_low", "1_mid", "2_high"]).astype(str)

for v in ["industry_pct", "road_density", "fire_density", "pop_density"]:
    if v in D and D[v].notna().all() and D[v].nunique() > 2:
        D[v + "_grp"] = make_groups(D.groupby(key)[v].transform("first"))
if EXPOSURE + "_grp" not in D:
    alt = next(v for v in ["industry_pct", "road_density", "fire_density"] if v + "_grp" in D)
    print(f"⚠ EXPOSURE = {EXPOSURE} ใช้ไม่ได้ (ไม่มีข้อมูล) → ใช้ {alt} แทน"); EXPOSURE = alt
D["urban"] = np.where(D.pop_density >= D.pop_density.median(), "urban", "rural")    # ตัวกวน: ความเป็นเมือง
G = EXPOSURE + "_grp"
display(D.groupby(G).agg(tambons=(key, "nunique"), cases=(OUTCOME, "sum"), person_years=("py", "sum")).assign(
    rate_per_1000=lambda t: t.cases / t.person_years * 1000).round(2))

## 2. หลักการ: อัตรา และ Rate Ratio

$$\text{rate} = \frac{\text{จำนวนเหตุการณ์}}{\text{person-time}}\qquad RR = \frac{a/PT_1}{b/PT_0}$$

| | เหตุการณ์ | person-time |
|---|---|---|
| กลุ่มสัมผัส (exposed) | a | PT₁ |
| กลุ่มอ้างอิง (unexposed) | b | PT₀ |

- **Wald CI:** $\ln RR \pm z\sqrt{1/a + 1/b}$
- **Exact CI:** ภายใต้เงื่อนไข a + b คงที่ $a \sim \text{Binomial}\big(a+b,\; \pi\big)$ โดย $\pi = \dfrac{RR\cdot PT_1}{RR\cdot PT_1 + PT_0}$ → ใช้ Clopper–Pearson บน π แล้วแปลงกลับ (เหมาะเมื่อจำนวนเหตุการณ์น้อย)
- **Rate Difference (RD):** $a/PT_1 - b/PT_0$, $SE = \sqrt{a/PT_1^2 + b/PT_0^2}$ — บอก "ส่วนเกิน" ในเชิงสาธารณสุข

In [ ]:
def rate_ratio(a, pt1, b, pt0, conf=CONF, per=1000):
    z = stats.norm.ppf(0.5 + conf / 2)
    r1, r0 = a / pt1, b / pt0; rr = r1 / r0
    se = np.sqrt(1 / a + 1 / b)
    lo, hi = np.exp(np.log(rr) - z * se), np.exp(np.log(rr) + z * se)
    n = a + b; al = 1 - conf
    plo = stats.beta.ppf(al / 2, a, n - a + 1) if a > 0 else 0.0
    phi = stats.beta.ppf(1 - al / 2, a + 1, n - a) if a < n else 1.0
    to_rr = lambda p: p / (1 - p) * pt0 / pt1
    p_exact = stats.binomtest(int(a), int(n), pt1 / (pt1 + pt0)).pvalue
    rd = (r1 - r0) * per; se_rd = np.sqrt(a / pt1**2 + b / pt0**2) * per
    return {"cases_exp": a, "PT_exp": pt1, "cases_ref": b, "PT_ref": pt0,
            f"rate_exp/{per}": r1 * per, f"rate_ref/{per}": r0 * per, "RR": rr,
            "Wald_lo": lo, "Wald_hi": hi, "exact_lo": to_rr(plo), "exact_hi": to_rr(phi) if phi < 1 else np.inf,
            "p_exact": p_exact, f"RD/{per}": rd, "RD_lo": rd - z * se_rd, "RD_hi": rd + z * se_rd}

# ตัวอย่างคำนวณด้วยมือ: กลุ่มสูงสุด vs กลุ่มอ้างอิง (รวมทุกปี)
lv = sorted(D[G].unique()); ref, top = lv[0], lv[-1]
a, pt1 = D.loc[D[G] == top, OUTCOME].sum(), D.loc[D[G] == top, "py"].sum()
b, pt0 = D.loc[D[G] == ref, OUTCOME].sum(), D.loc[D[G] == ref, "py"].sum()
print(f"{EXPOSURE}: {top} vs {ref}  |  a = {a:,.0f} / {pt1:,.0f} PY ,  b = {b:,.0f} / {pt0:,.0f} PY")
pd.Series(rate_ratio(a, pt1, b, pt0)).round(4)

## 3. Crude RR ทุกระดับการสัมผัส และ dose–response
- เทียบแต่ละระดับกับกลุ่มอ้างอิง
- **Trend test:** Poisson regression ที่ใส่ลำดับกลุ่ม (0, 1, 2) เป็นตัวแปรต่อเนื่อง — p < 0.05 แสดงว่า RR เพิ่ม/ลดตามระดับการสัมผัส

In [ ]:
rows = []
for exp in ["industry_pct", "road_density", "fire_density"]:
    g = exp + "_grp"
    if g not in D: continue
    lv = sorted(D[g].unique())
    b_, p0_ = D.loc[D[g] == lv[0], OUTCOME].sum(), D.loc[D[g] == lv[0], "py"].sum()
    for l in lv[1:]:
        r = rate_ratio(D.loc[D[g] == l, OUTCOME].sum(), D.loc[D[g] == l, "py"].sum(), b_, p0_)
        rows.append({"exposure": exp, "level": l, "vs": lv[0], **{k: r[k] for k in ["RR", "Wald_lo", "Wald_hi", "p_exact"]}})
    D["_ord"] = D[g].str[0].astype(int)
    trend = smf.glm(f"Q('{OUTCOME}') ~ _ord", data=D, family=sm.families.Poisson(), offset=np.log(D.py)).fit(scale="X2")
    rows[-1]["p_trend (quasi-Poisson)"] = trend.pvalues["_ord"]
CRUDE = pd.DataFrame(rows); display(CRUDE.round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
for k, (exp, t) in enumerate(CRUDE.groupby("exposure")):
    y = np.arange(len(t)) + k * 3
    ax.errorbar(t.RR, y, xerr=[t.RR - t.Wald_lo, t.Wald_hi - t.RR], fmt="o", capsize=3, label=exp)
    for yy, l in zip(y, t.level): ax.text(t.Wald_hi.max() * 1.05, yy, l, fontsize=7, va="center")
ax.axvline(1, color="k", ls="--"); ax.set_xscale("log"); ax.set_yticks([]); ax.set_xlabel("crude rate ratio (log scale)")
ax.legend(fontsize=7); ax.set_title(f"Crude RR — outcome: {OUTCOME}", fontsize=9); plt.show()

## 4. ควบคุมตัวกวนด้วย Mantel–Haenszel RR

**ตัวกวน (confounder)** ที่สำคัญ: ความเป็นเมือง (ตำบลเมืองมีทั้งถนน/โรงงานมากและเข้าถึงโรงพยาบาลง่าย → พบผู้ป่วยมาก) และปีที่บันทึก

$$RR_{MH} = \frac{\sum_k a_k\,PT_{0k}/PT_k}{\sum_k b_k\,PT_{1k}/PT_k}$$

ความแปรปรวนของ $\ln RR_{MH}$ ตาม Greenland & Robins (1985) · **Homogeneity test:** ถ้า RR ต่างกันมากระหว่างชั้น (p < 0.05) อาจเป็น *effect modification* — ควรรายงานแยกชั้นแทนค่ารวม

In [ ]:
def mantel_haenszel(df, exp_col, top, ref, strata, outcome=OUTCOME, conf=CONF):
    z = stats.norm.ppf(0.5 + conf / 2); num = den = var_num = 0; lnrr, w, tab = [], [], []
    for s, d in df[df[exp_col].isin([top, ref])].groupby(strata):
        a = d.loc[d[exp_col] == top, outcome].sum(); b = d.loc[d[exp_col] == ref, outcome].sum()
        n1 = d.loc[d[exp_col] == top, "py"].sum(); n0 = d.loc[d[exp_col] == ref, "py"].sum(); T = n1 + n0
        if n1 == 0 or n0 == 0: continue
        num += a * n0 / T; den += b * n1 / T; var_num += n1 * n0 * (a + b) / T**2
        tab.append({"stratum": s, "a": a, "PT1": n1, "b": b, "PT0": n0, "RR_stratum": (a / n1) / (b / n0) if b > 0 else np.nan})
        if a > 0 and b > 0: lnrr.append(np.log((a / n1) / (b / n0))); w.append(1 / (1 / a + 1 / b))
    rr = num / den; se = np.sqrt(var_num / (num * den))
    lnrr, w = np.array(lnrr), np.array(w); iv = (w * lnrr).sum() / w.sum()
    Q = (w * (lnrr - iv) ** 2).sum(); p_het = 1 - stats.chi2.cdf(Q, len(w) - 1) if len(w) > 1 else np.nan
    return rr, np.exp(np.log(rr) - z * se), np.exp(np.log(rr) + z * se), p_het, pd.DataFrame(tab)

lv = sorted(D[G].unique()); ref, top = lv[0], lv[-1]
crude = rate_ratio(D.loc[D[G] == top, OUTCOME].sum(), D.loc[D[G] == top, "py"].sum(),
                   D.loc[D[G] == ref, OUTCOME].sum(), D.loc[D[G] == ref, "py"].sum())
rows = [{"adjusted for": "— (crude)", "RR": crude["RR"], "lo": crude["Wald_lo"], "hi": crude["Wald_hi"], "p_homogeneity": np.nan}]
for strata in [["year"], ["urban"], ["urban", "year"]]:
    rr, lo, hi, ph, tab = mantel_haenszel(D, G, top, ref, strata)
    rows.append({"adjusted for": " + ".join(strata), "RR": rr, "lo": lo, "hi": hi, "p_homogeneity": ph})
    if strata == ["urban"]: display(tab.round(4))
MH = pd.DataFrame(rows); print(f"{EXPOSURE}: {top} vs {ref}"); display(MH.round(4))
print("ถ้า RR หลังปรับต่างจาก crude มาก (> ~10%) แสดงว่าตัวแปรนั้นเป็นตัวกวนที่สำคัญ")

## 5. Poisson regression มี offset → adjusted IRR

$$\ln E[Y_{it}] = \ln(PT_{it}) + \beta_0 + \beta_1\,\text{Exposure}_i + \beta_2\,\text{Year}_t + \beta_3\,\ln(\text{PopDensity}_i)$$

- $\ln(PT)$ คือ **offset** (สัมประสิทธิ์ถูกบังคับ = 1) → $e^{\beta_1}$ = **adjusted IRR**
- ตรวจ **overdispersion:** Pearson χ²/df ≫ 1 → SE ของ Poisson เล็กเกินจริง → ใช้ quasi-Poisson, **Negative binomial** หรือ **cluster-robust SE** (ตำบลเดียวกันถูกนับซ้ำหลายปี)

In [ ]:
D["log_pd"] = np.log(D.pop_density); D["log_py"] = np.log(D.py)
f = f"Q('{OUTCOME}') ~ C({G}) + C(year) + log_pd"
pois = smf.glm(f, data=D, family=sm.families.Poisson(), offset=D.log_py).fit()
disp = pois.pearson_chi2 / pois.df_resid
print(f"Pearson χ²/df = {disp:.1f}  ({'overdispersion — ต้องปรับ' if disp > 1.5 else 'พอใช้ได้'})")
qp = smf.glm(f, data=D, family=sm.families.Poisson(), offset=D.log_py).fit(scale="X2")
cl = smf.glm(f, data=D, family=sm.families.Poisson(), offset=D.log_py).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(D[key])[0]})
try:
    nbm = smf.negativebinomial(f, data=D, offset=D.log_py).fit(disp=0, maxiter=200)
except Exception as e:
    nbm = None; print("NB fit ไม่สำเร็จ:", str(e)[:80])

def irr_table(m, name):
    ci = m.conf_int(alpha=1 - CONF)
    t = pd.DataFrame({"IRR": np.exp(m.params), "lo": np.exp(ci[0]), "hi": np.exp(ci[1]), "p": m.pvalues})
    t = t[t.index.str.contains(G) | (t.index == "log_pd")]; t.insert(0, "model", name); return t
IRR = pd.concat([irr_table(pois, "Poisson (naive SE)"), irr_table(qp, "quasi-Poisson"),
                 irr_table(cl, "Poisson + cluster-robust SE")] + ([irr_table(nbm, "Negative binomial")] if nbm is not None else []))
display(IRR.round(4))
print("สังเกต: IRR ของ Poisson/quasi/cluster เท่ากัน ต่างกันเฉพาะช่วงความเชื่อมั่น — ใช้ cluster-robust หรือ NB ในการรายงาน")

## 6. Forest plot: adjusted IRR (กลุ่มสูงสุด vs อ้างอิง) แยกรายโรค

In [ ]:
rows = []
for dis in DISEASES:
    if D[dis].sum() < 20: continue
    try:
        m = smf.glm(f"Q('{dis}') ~ C({G}) + C(year) + log_pd", data=D, family=sm.families.Poisson(),
                    offset=D.log_py).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(D[key])[0]})
        term = [t for t in m.params.index if G in t and top in t][0]; ci = m.conf_int().loc[term]
        rows.append({"disease": dis, "cases": int(D[dis].sum()), "IRR": np.exp(m.params[term]),
                     "lo": np.exp(ci[0]), "hi": np.exp(ci[1]), "p": m.pvalues[term]})
    except Exception as e:
        print(dis, "fit ไม่ได้:", str(e)[:60])
FOREST = pd.DataFrame(rows).sort_values("IRR")
fig, ax = plt.subplots(figsize=(7, 0.45 * len(FOREST) + 1.5))
y = np.arange(len(FOREST))
ax.errorbar(FOREST.IRR, y, xerr=[FOREST.IRR - FOREST.lo, FOREST.hi - FOREST.IRR], fmt="s", color="k", capsize=3)
ax.axvline(1, ls="--", color="grey"); ax.set_xscale("log")
ax.set_yticks(y, [f"{d} (n={c:,})" for d, c in zip(FOREST.disease, FOREST.cases)], fontsize=8)
ax.set_xlabel(f"adjusted IRR: {EXPOSURE} {top} vs {ref} (year, log pop density; cluster-robust 95% CI)")
plt.tight_layout(); plt.show()
display(FOREST.round(4))

## 7. บันทึกผล

In [ ]:
D.to_csv(f"{OUTDIR}/rr_tambon_year_data.csv", index=False)
CRUDE.to_csv(f"{OUTDIR}/rr_crude.csv", index=False); MH.to_csv(f"{OUTDIR}/rr_mantel_haenszel.csv", index=False)
IRR.to_csv(f"{OUTDIR}/rr_poisson_irr.csv"); FOREST.to_csv(f"{OUTDIR}/rr_forest_by_disease.csv", index=False)
json.dump({**DATA_SOURCES, "hdc": HDC_SRC, "hdc_last_processed": last_proc}, open(f"{OUTDIR}/rr_sources.json", "w"),
          ensure_ascii=False, indent=1)
print("บันทึกแล้วใน", OUTDIR)

## 8. ข้อจำกัดและการตีความ (สำคัญมากสำหรับข้อมูลระดับพื้นที่)
1. **Ecological fallacy** — RR นี้เป็นความสัมพันธ์ระดับ "ตำบล" ไม่ได้บอกว่าบุคคลที่สัมผัสมากป่วยมากกว่า
2. **ตัวเศษ (HDC) คือจำนวนครั้ง/รายที่มารับบริการและถูกบันทึก** — ขึ้นกับการเข้าถึงบริการ ความครบถ้วนของการบันทึก และสถานบริการที่ผู้ป่วยไปใช้ (ผู้ป่วยที่ไปรักษานอกจังหวัดไม่ถูกนับ)
3. **ตัวหาร (WorldPop 2020)** เป็นค่าประมาณจากแบบจำลองและคงที่ทุกปี — ถ้ามีประชากรรายตำบลจากทะเบียนราษฎร์ควรใช้แทน
4. **Exposure เป็น proxy** (พื้นที่โรงงาน/ถนน/จุดความร้อน) ไม่ใช่ความเข้มข้น PM2.5 ที่บุคคลได้รับ
5. **ข้อมูลซ้ำในตำบลเดียวกันหลายปี** ไม่อิสระ → ใช้ cluster-robust SE หรือโมเดล mixed effects
6. ใช้เฉพาะ **ปีงบประมาณที่ครบ 12 เดือน** (ตรวจ `hdc_last_processed`) มิฉะนั้นอัตราของปีล่าสุดจะต่ำผิดปกติ

**แบบฝึกหัด**
1. เปลี่ยน `EXPOSURE = "road_density"` — สอดคล้องกับประเด็นประชากรริมถนน
2. เปลี่ยน `OUTCOME = "COPD"` หรือ `"Asthma"` แล้วเทียบกับ `"กลุ่มโรคผิวหนังอักเสบ"` (โรคที่คาดว่าสัมพันธ์กับ PM2.5 น้อยกว่า → ใช้เป็น *negative control outcome*)
3. เพิ่มตัวกวน `fire_density` ในสูตร Poisson แล้วดูว่า IRR ของ `industry_pct` เปลี่ยนเท่าไร

**อ้างอิง:** Rothman, Greenland & Lash (2008) *Modern Epidemiology* 3rd ed. · Greenland & Robins (1985) *Biometrics* 41:55 · Breslow & Day (1987) *Statistical Methods in Cancer Research Vol. II* (IARC) · Cameron & Trivedi (2013) *Regression Analysis of Count Data* · Lipsitch, Tchetgen & Cohen (2010) *Epidemiology* 21:383 (negative controls) · MOPH Open Data / HDC · WorldPop (2018) · OCHA COD-AB Thailand